# PROBLEM 2: CHARACTER-LEVEL NAME GENERATION USING RNN VARIANTS

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import string

## TASK-0: DATASET

In [5]:
# Load the data
def load_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        names = [line.strip().lower() for line in f if line.strip()]
    return names

names = load_data("TrainingNames.txt")

# Build vocabulary
chars = sorted(list(set(''.join(names))))
chars.append('<EOS>')

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

vocab_size = len(chars)

In [4]:
# Additional functions
def encode(name):
    return [stoi[ch] for ch in name] + [stoi['<EOS>']]

def decode(indices):
    name = ''
    for i in indices:
        if itos[i] == '<EOS>':
            break
        name += itos[i]
    return name

## TASK-1: MODEL IMPLEMENTATION

In [6]:
# 1. Vanilla Recurrent Neural Network (RNN)

class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        # Hidden state weights
        self.W_xh = nn.Linear(vocab_size, hidden_size, bias=False)
        self.W_hh = nn.Linear(hidden_size, hidden_size)
        # Output weights
        self.W_hy = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h):
        # Hidden state update
        h = torch.tanh(self.W_xh(x) + self.W_hh(h))
        # Output
        y = self.W_hy(h)
        return y, h

    def init_hidden(self):
        return torch.zeros(1, self.hidden_size)

In [7]:
# 2. Bidirectional Long Short-Term Memory (BLSTM)

class BLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=vocab_size,
            hidden_size=hidden_size,
            bidirectional=True,
            batch_first=True
        )

        self.fc = nn.Linear(2 * hidden_size, vocab_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out

In [8]:
# 3. RNN with Basic Attention Mechanism

class AttentionRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size

        self.W_xh = nn.Linear(vocab_size, hidden_size, bias=False)
        self.W_hh = nn.Linear(hidden_size, hidden_size)

        self.W_hy = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()

        h = torch.zeros(batch_size, self.hidden_size)
        outputs = []

        # RNN forward
        for t in range(seq_len):
            xt = x[:, t, :]
            h = torch.tanh(self.W_xh(xt) + self.W_hh(h))
            outputs.append(h.unsqueeze(1))

        outputs = torch.cat(outputs, dim=1)

        # Attention
        scores = torch.bmm(outputs, outputs.transpose(1, 2))
        attn_weights = torch.softmax(scores, dim=-1)

        # Context vector
        context = torch.bmm(attn_weights, outputs)

        y = self.W_hy(context)

        return y

In [9]:
# Train function
def train_model(model, names, epochs=25, lr=0.001):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss = 0

        for name in names:
            encoded = encode(name)

            # For VanillaRNN
            if isinstance(model, VanillaRNN):
                h = model.init_hidden()
                loss = 0

                for t in range(len(encoded) - 1):
                    x = torch.zeros(1, vocab_size)
                    x[0][encoded[t]] = 1

                    y = torch.tensor([encoded[t + 1]])

                    out, h = model(x, h)
                    loss += loss_fn(out, y)

            # For BLSTM
            elif isinstance(model, BLSTM):
                x = torch.zeros(1, len(encoded) - 1, vocab_size)
                y = torch.tensor(encoded[1:])

                for t in range(len(encoded) - 1):
                    x[0][t][encoded[t]] = 1

                out = model(x)  # (1, seq_len, vocab)
                out = out.view(-1, vocab_size)

                loss = loss_fn(out, y)

            # For AttentionRNN
            elif isinstance(model, AttentionRNN):
                x = torch.zeros(1, len(encoded) - 1, vocab_size)
                y = torch.tensor(encoded[1:])

                for t in range(len(encoded) - 1):
                    x[0][t][encoded[t]] = 1

                out = model(x)  # (1, seq_len, vocab)
                out = out.view(-1, vocab_size)

                loss = loss_fn(out, y)

            # backpropagation
            optimizer.zero_grad()
            loss.backward()

            # gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)

            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.2f}")

In [10]:
def sample(logits, temperature=0.7, top_k=5):
    logits = logits / temperature
    probs = torch.softmax(logits, dim=-1)

    # Top-k filtering
    top_probs, top_idx = torch.topk(probs, top_k)
    top_probs = top_probs / torch.sum(top_probs)

    idx = torch.multinomial(top_probs, 1).item()
    return top_idx[idx].item()

In [11]:
# Generation function
def generate_name(model, start_char=None, max_len=12):
    model.eval()

    if start_char is None:
        start_char = random.choice(string.ascii_lowercase)

    name = start_char

    with torch.no_grad():

        # For VanillaRNN
        if isinstance(model, VanillaRNN):
            h = model.init_hidden()

            # feed initial characters
            for ch in name[:-1]:
                x = torch.zeros(1, vocab_size)
                x[0][stoi[ch]] = 1
                _, h = model(x, h)

            current_char = name[-1]

            for _ in range(max_len):
                x = torch.zeros(1, vocab_size)
                x[0][stoi[current_char]] = 1

                out, h = model(x, h)
                idx = sample(out[0])

                if itos[idx] == '<EOS>':
                    break

                current_char = itos[idx]
                name += current_char

        # For BLSTM and AttentionRNN
        else:
            for _ in range(max_len):
                encoded = [stoi[ch] for ch in name if ch in stoi]

                x = torch.zeros(1, len(encoded), vocab_size)
                for t, idx in enumerate(encoded):
                    x[0][t][idx] = 1

                out = model(x)  # (1, seq_len, vocab)
                logits = out[0, -1]

                idx = sample(logits)

                if itos[idx] == '<EOS>':
                    break

                name += itos[idx]

    return name

## TASK-2: QUANTITATIVE EVALUATION

In [14]:
def save_model(model, path):
    torch.save(model.state_dict(), path)

def load_model(model, path):
    model.load_state_dict(torch.load(path))
    model.eval()
    return model

In [15]:
def generate_names(model, n=1000):
    generated = []
    for _ in range(n):
        name = generate_name(model)
        generated.append(name)
    return generated

In [17]:
# Metrics
def compute_metrics(generated_names, training_names):
    training_set = set(training_names)
    generated_set = set(generated_names)

    # Novelty Rate
    novel_count = sum(1 for name in generated_names if name not in training_set)
    novelty_rate = (novel_count / len(generated_names)) * 100

    # Diversity
    diversity = len(generated_set) / len(generated_names)

    return novelty_rate, diversity

In [18]:
hidden_size = 128

# Initialize models
rnn_model = VanillaRNN(vocab_size, hidden_size)
blstm_model = BLSTM(vocab_size, hidden_size)
attn_model = AttentionRNN(vocab_size, hidden_size)

In [19]:
# Training and quantitvative evaluation of rnn
print("\n===== VANILLA RNN =====")

# Train
train_model(rnn_model, names, epochs=25, lr=0.002)

# Save
save_model(rnn_model, "rnn_model.pth")

# Generate samples
rnn_samples = generate_names(rnn_model, 20)
print("Sample Names:", rnn_samples)

# Evaluate
rnn_generated = generate_names(rnn_model, 1000)
rnn_novelty, rnn_diversity = compute_metrics(rnn_generated, names)

print(f"Novelty: {rnn_novelty:.2f}%")
print(f"Diversity: {rnn_diversity:.2f}")


===== VANILLA RNN =====
Epoch 1/25, Loss: 25933.80
Epoch 2/25, Loss: 17011.58
Epoch 3/25, Loss: 12864.11
Epoch 4/25, Loss: 10708.37
Epoch 5/25, Loss: 9489.56
Epoch 6/25, Loss: 8712.25
Epoch 7/25, Loss: 8188.49
Epoch 8/25, Loss: 7813.15
Epoch 9/25, Loss: 7565.57
Epoch 10/25, Loss: 7337.94
Epoch 11/25, Loss: 7236.24
Epoch 12/25, Loss: 7091.04
Epoch 13/25, Loss: 7000.33
Epoch 14/25, Loss: 6907.96
Epoch 15/25, Loss: 6834.12
Epoch 16/25, Loss: 6699.91
Epoch 17/25, Loss: 6696.86
Epoch 18/25, Loss: 6665.55
Epoch 19/25, Loss: 6489.85
Epoch 20/25, Loss: 6438.70
Epoch 21/25, Loss: 6470.01
Epoch 22/25, Loss: 6495.13
Epoch 23/25, Loss: 6428.54
Epoch 24/25, Loss: 6306.44
Epoch 25/25, Loss: 6377.15
Sample Names: ['ojas mehta', 'brijesh baner', 'aditi roy', 'raghav reddy', 'umesh dutta', 'vidya mishra', 'bhavana rao', 'chetan kurian', 'mohan gupta', 'jaya roy', 'eshan menon', 'pradeep dutta', 'ekta pillai', 'jaya joshi', 'zehra patel', 'quasar george', 'waseem malhot', 'vinod menon', 'brijesh baner'

In [20]:
# Training and quantitvative evaluation of blstm
print("\n===== BLSTM =====")

# Train
train_model(blstm_model, names, epochs=25, lr=0.001)

# Save
save_model(blstm_model, "blstm_model.pth")

# Generate samples
blstm_samples = generate_names(blstm_model, 20)
print("Sample Names:", blstm_samples)

# Evaluate
blstm_generated = generate_names(blstm_model, 1000)
blstm_novelty, blstm_diversity = compute_metrics(blstm_generated, names)

print(f"Novelty: {blstm_novelty:.2f}%")
print(f"Diversity: {blstm_diversity:.2f}")


===== BLSTM =====
Epoch 1/25, Loss: 1155.29
Epoch 2/25, Loss: 39.08
Epoch 3/25, Loss: 7.65
Epoch 4/25, Loss: 2.76
Epoch 5/25, Loss: 1.30
Epoch 6/25, Loss: 0.68
Epoch 7/25, Loss: 0.37
Epoch 8/25, Loss: 0.21
Epoch 9/25, Loss: 0.12
Epoch 10/25, Loss: 0.07
Epoch 11/25, Loss: 0.04
Epoch 12/25, Loss: 0.02
Epoch 13/25, Loss: 0.01
Epoch 14/25, Loss: 0.01
Epoch 15/25, Loss: 0.00
Epoch 16/25, Loss: 0.00
Epoch 17/25, Loss: 0.00
Epoch 18/25, Loss: 0.00
Epoch 19/25, Loss: 0.00
Epoch 20/25, Loss: 9.73
Epoch 21/25, Loss: 0.19
Epoch 22/25, Loss: 0.07
Epoch 23/25, Loss: 0.04
Epoch 24/25, Loss: 0.03
Epoch 25/25, Loss: 0.02
Sample Names: ['s', 'ewa', 'mmin', 'yall ', 'haxii', 'r', 'z', 'ggh', 'vyy', 'jya', 'xika ', 'tii', 'shin', 'max', 'dyaxi f', 'fa', 'y', 'oz', 'axeezzeba', 'kllll ma']
Novelty: 100.00%
Diversity: 0.61


In [21]:
# Training and quantitvative evaluation of RNN with attention
print("\n===== ATTENTION RNN =====")

# Train
train_model(attn_model, names, epochs=25, lr=0.0015)

# Save
save_model(attn_model, "attn_model.pth")

# Generate samples
attn_samples = generate_names(attn_model, 20)
print("Sample Names:", attn_samples)

# Evaluate
attn_generated = generate_names(attn_model, 1000)
attn_novelty, attn_diversity = compute_metrics(attn_generated, names)

print(f"Novelty: {attn_novelty:.2f}%")
print(f"Diversity: {attn_diversity:.2f}")


===== ATTENTION RNN =====
Epoch 1/25, Loss: 2251.05
Epoch 2/25, Loss: 1512.91
Epoch 3/25, Loss: 1150.23
Epoch 4/25, Loss: 954.76
Epoch 5/25, Loss: 835.69
Epoch 6/25, Loss: 759.95
Epoch 7/25, Loss: 705.04
Epoch 8/25, Loss: 666.92
Epoch 9/25, Loss: 636.82
Epoch 10/25, Loss: 610.34
Epoch 11/25, Loss: 592.24
Epoch 12/25, Loss: 575.61
Epoch 13/25, Loss: 561.31
Epoch 14/25, Loss: 547.63
Epoch 15/25, Loss: 538.18
Epoch 16/25, Loss: 530.14
Epoch 17/25, Loss: 516.26
Epoch 18/25, Loss: 511.31
Epoch 19/25, Loss: 504.51
Epoch 20/25, Loss: 496.36
Epoch 21/25, Loss: 492.97
Epoch 22/25, Loss: 485.96
Epoch 23/25, Loss: 477.84
Epoch 24/25, Loss: 481.14
Epoch 25/25, Loss: 478.43
Sample Names: ['pratibhi ghos', 'abhay kulkarn', 'kavya keenan', 'abhay vyas', 'chitra shetty', 'farah joshi', 'urvashi rao', 'aditi cherian', 'poonam iyer', 'bhavana thoma', 'wasim singh', 'kiran kurian', 'ekta deshmukh', 'tanya singh', 'jagdish cheri', 'rashmi mishra', 'mohan gupta', 'farah menon', 'gautam kurian', 'nisha seb

In [22]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("No of parameters of Vanilla RNN:", count_params(rnn_model))
print("No of parameters of BLSTM:", count_params(blstm_model))
print("No of parameters of RNN with attention:", count_params(attn_model))

No of parameters of Vanilla RNN: 23708
No of parameters of BLSTM: 168988
No of parameters of RNN with attention: 23708


In [23]:
# Evaluate
print("\nFINAL COMPARISON:")
print(f"RNN       - Novelty: {rnn_novelty:.2f}% | Diversity: {rnn_diversity:.2f}")
print(f"BLSTM     - Novelty: {blstm_novelty:.2f}% | Diversity: {blstm_diversity:.2f}")
print(f"Attention - Novelty: {attn_novelty:.2f}% | Diversity: {attn_diversity:.2f}")


FINAL COMPARISON:
RNN       -> Novelty: 59.40% | Diversity: 0.47
BLSTM     -> Novelty: 100.00% | Diversity: 0.61
Attention -> Novelty: 55.50% | Diversity: 0.49


## TASK-3: QUALITATIVE ANALYSIS

In [26]:
rnn_samples = generate_names(rnn_model, 10)
print("RNN sample Names:", rnn_samples)

blstm_samples = generate_names(blstm_model, 10)
print("BLSTM sample Names:", blstm_samples)

attn_samples = generate_names(attn_model, 10)
print("AttentoinRNN Sample Names:", attn_samples)

RNN sample Names: ['vivek shetty', 'quasar fernan', 'quasar george', 'yamini cheria', 'vidya mishra', 'harini singh', 'quasar kurian', 'umesh deshmuk', 'xavier das', 'kunal kulkarn']
BLSTM sample Names: ['qa', 'ullllisimm', 'tis', 'yeeezzeeb', 'gh', 'wal', 'feewa f', 'tinin j', 'hax', 'ryyyy f']
AttentoinRNN Sample Names: ['kiran kurian', 'gaurav joshi', 'naveen mishra', 'geeta gupta', 'bharti kaur', 'urean malhotr', 'tanmay teshau', 'chitra verma', 'madhuri menon', 'arjen mehta']
